# Compare NSQIP model performance to ours for

Outcomes:
- SSI
- Pneumonia
- Any
- Serious
- Unplanned Reoper

Metrics:
- AUROC (DeLongs)
- AUPRC/Avg Precision (non-parametric Bootstrap)
- Brier (non-parametric Bootstrap)
- ***NOT ICI*** bc we use custom bin thresholds for our models, wouldn't be comparable

***Note:*** Will compare all models here but only present interface-selected comparisons in manuscript

## Set Up

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.nsqip_eval import fill_model_res_dict_update, plot_roc_prc, format_p
import pandas as pd
import numpy as np

In [ ]:
# Use calibrated models
mdoel_imp_dir = BASE_PATH / "models" / "calibrated"
MODEL_LIST = [
    "svc",
    "lr",
    "nn",
    "lgbm",
    "xgb",
    "stack",
]
outcome_imp_dir = BASE_PATH / "data"
OUTCOME_LIST = [
    "ssi",
    "reoper",
    "any",
    "serious",
    "pneumo",
    # BLEED not available on NSQIP site
]
proba_dir = BASE_PATH / "data" / "nsqip_data" / "processed_probs"
NSQIP_RES_DIR = BASE_PATH / "results" / "nsqip_eval"

N_BOOT = 5000
N_PERM = 10000

## Sig testing

Construct result dict skeleton

In [ ]:
outcome_res_dict = {}
for outcome_path in proba_dir.iterdir():
    outcome_name = outcome_path.stem
    print(f"{outcome_name}...")
    df = pd.read_parquet(outcome_path)
    df = df.reset_index(drop=False)
    y_true = df["y_true"]
    y_proba_nsqip = df["y_proba_nsqip"]
    outcome_res_dict[outcome_name] = fill_model_res_dict_update(
        proba_df=df,
        y_true=y_true,
        y_proba_nsqip=y_proba_nsqip,
        model_list=MODEL_LIST,
        n_boot=N_BOOT,
        n_perm=N_PERM,
        rand_state=SEED,
    )
    ### PLOT + export ROC ###
    plot_roc_prc(
        y_true=y_true,
        y_proba=y_proba_nsqip,
        ## Just choosing svc arbitrarily here
        # AUROC & AUPRC CIs are same accross all models
        roc_val_ci_str=outcome_res_dict[outcome_name]["svc"]["auroc"]["ci_str"],
        prc_val_ci_str=outcome_res_dict[outcome_name]["svc"]["avg_prec"]["ci_str"],
        outcome_name=outcome_name,
        res_dir=NSQIP_RES_DIR,
    )

In [ ]:
df_rows = []
for outcome_name, dict0 in outcome_res_dict.items():
    print(outcome_name)
    for model_name, dict1 in dict0.items():
        print(f"\t{model_name}")
        for metric_name, dict2 in dict1.items():
            # Ensure obsv val & CI val match, ultimately use obsv val
            obsv_nsqip = dict2["nsqip_val"]
            ci_str = dict2["ci_str"]
            ci_str_split = ci_str.split(" ")
            ci_val = float(ci_str_split[0])
            just_cis = "".join(ci_str_split[1:])
            assert np.isclose(
                ci_val, obsv_nsqip, atol=0.0005
            ), f"{ci_val}--{obsv_nsqip}"
            ## rest of results
            p_val = dict2["p_val"]
            p_val_str = format_p(p_val)
            diff = round(dict2["us_val"] - obsv_nsqip, 3)
            df_rows.append(
                {
                    "outcome_name": outcome_name,
                    "model_name": model_name,
                    "metric": metric_name,
                    "val (CIs)": f"{obsv_nsqip:.3f} {just_cis}",
                    "p": p_val_str,
                    "observed diff": diff,
                }
            )
result_df = pd.DataFrame(df_rows)

Export

In [ ]:
res_df_path = NSQIP_RES_DIR / "raw_table.xlsx"

if res_df_path.exists():
    res_df_path.unlink()
res_df_path.parent.mkdir(exist_ok=True, parents=True)
result_df.to_excel(res_df_path)